In [14]:
import pandas as pd
import numpy as np

# Carica il dataset
df = pd.read_csv('Data/train_data_30.csv')

# Funzione per aggiungere rumore ai dati
def add_noise(data, noise_level=0.01):
    noise = np.random.randn(*data.shape) * noise_level
    return data + noise

# Funzione per shiftare i dati
def shift_data(data, shift_max=2):
    shift_val = np.random.randint(-shift_max, shift_max)
    return np.roll(data, shift_val, axis=0)

# Funzione per effettuare time stretching
def time_stretch(data, stretch_factor=1.1):
    indices = np.round(np.arange(0, len(data), stretch_factor))
    indices = indices[indices < len(data)].astype(int)
    return data[indices]

# Funzione per effettuare pitch shifting
def pitch_shift(data, shift_max=2):
    return np.roll(data, shift_max)

# Funzione per aumentare i dati
def augment_data(df):
    augmented_df = pd.DataFrame()
    for label in df['label'].unique():
        subset = df[df['label'] == label]
        for _ in range(10):  # Crea 10 campioni aumentati per ogni campione originale
            noisy_data = add_noise(subset.iloc[:, 1:-1].values)
            shifted_data = shift_data(noisy_data)
            stretched_data = time_stretch(shifted_data)
            pitch_shifted_data = pitch_shift(stretched_data)
            augmented_subset = pd.DataFrame(pitch_shifted_data, columns=subset.columns[1:-1])
            augmented_subset.insert(0, 'filename', subset['filename'].values[:len(augmented_subset)])  # Aggiungi il nome del file come prima colonna
            augmented_subset['label'] = label
            augmented_df = pd.concat([augmented_df, augmented_subset], ignore_index=True)
    return augmented_df

# Aumenta i dati
augmented_df = augment_data(df)

# Combina i dati originali e aumentati
combined_df = pd.concat([df, augmented_df], ignore_index=True)

# Salva il dataset aumentato in un nuovo file CSV
combined_df.to_csv('Data/augmented_train_data_30.csv', index=False)

print("Data augmentation completata e salvata in 'augmented_train_data_30.csv'.")

Data augmentation completata e salvata in 'augmented_train_data_30.csv'.
